In [ ]:
import cv2
import numpy as np
import torch
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

# Ensure you have imported your ColorizationVAE class and Config from your training script
# For example:
# from your_autoencoder_module import ColorizationVAE, Config

# For this example, we'll assume ColorizationVAE and Config are already defined in the namespace

def load_model(model_path, latent_dim=128):
    model = ColorizationVAE(latent_dim=latent_dim)
    state_dict = torch.load(model_path, map_location=Config.device)
    model.load_state_dict(state_dict)
    model = model.to(Config.device)
    model.eval()
    return model

def run_inference(model, image_path):
    # Load image using OpenCV
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Failed to load image from {image_path}")
    # Convert from BGR to RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(image)

    # Define the transforms (should match what you used for inference)
    # For input (grayscale) transformation:
    transform_input = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
    ])
    # For target (color) transformation (for visualization, if needed):
    transform_target = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
        transforms.ToTensor(),
    ])

    # Prepare the input tensor (grayscale)
    gray_tensor = transform_input(np.array(pil_image)).unsqueeze(0)  # add batch dim
    gray_tensor = gray_tensor.to(Config.device)

    # Run inference through the autoencoder
    with torch.no_grad():
        output, _, _ = model(gray_tensor)
    
    # Move the outputs back to CPU and convert to numpy arrays
    output = output.squeeze(0).cpu()  # shape (3, H, W)
    output_img = output.permute(1, 2, 0).numpy()  # convert to HxWx3

    # For comparison, also get the grayscale input as image
    gray_img = gray_tensor.squeeze(0).cpu().numpy()  # shape (1, H, W)
    gray_img = np.squeeze(gray_img)  # shape (H, W)

    # Optionally, get the original color target (if available) for comparison:
    color_tensor = transform_target(np.array(pil_image)).unsqueeze(0)
    color_img = color_tensor.squeeze(0).numpy()
    color_img = color_img.transpose(1, 2, 0)

    return gray_img, output_img, color_img

In [ ]:
# Define the path to your trained model and a sample image.
model_path = Config.WEIGHTS_DIR / "fox_vae_model.pth"  # adjust model name/path as needed
sample_image_path = Config.DATA_DIR / "fox" / "test" / "tiger_101.jpg"  # adjust sample image path
# Load the model.
model = load_model(model_path, latent_dim=128)
# Run inference on the sample image.
gray_img, output_img, color_img = run_inference(model, sample_image_path)
# Plot the input grayscale, the output (colorized) image, and the original color image.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(gray_img, cmap="gray")
axes[0].set_title("Input (Grayscale)")
axes[0].axis("off")
axes[1].imshow(output_img)
axes[1].set_title("Output (Colorized)")
axes[1].axis("off")
axes[2].imshow(color_img)
axes[2].set_title("Original Color")
axes[2].axis("off")
plt.tight_layout()
plt.show()